# Task 4 — Direct Kafka-to-Neo4j ingestion

```{admonition} Evidence summary
:class: note

| Field | Value |
|---|---|
| Evidence source | Stage 2 connector/query artifacts plus Stage 3 UI evidence |
| Command | `bash scripts/capture_connector_evidence.sh`; `bash scripts/capture_store_evidence.sh` |
| Route | `cpg.nodes,cpg.edges` → Neo4j Kafka Connector → Neo4j |
| Run date | `2026-07-23` |
```

## Approach and reasoning

Kafka Connect writes graph events directly to Neo4j; Spark is deliberately absent
from this route. Connector Cypher uses `MERGE` on stable IDs, while verification
queries compare database counts with unique Kafka IDs and reject duplicates.

## Key implementation excerpts

```{literalinclude} ../neo4j/verification.cypher
:language: cypher
:lines: 1-19
:caption: Node, edge, duplicate, and placeholder verification
```

## Executed evidence


In [1]:
from pathlib import Path
import json

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "screenshots/stage2_manifest.json").exists()
)
manifest = json.loads((ROOT / "screenshots/stage2_manifest.json").read_text())
neo4j = manifest["metrics"]["neo4j"]
status = json.loads((ROOT / "screenshots/kafka/connector_status.json").read_text())
connector = json.loads((ROOT / "neo4j/sink_connector.json").read_text())["config"]

raw = {
    name: (ROOT / "screenshots/neo4j" / name).read_text().strip()
    for name in (
        "node_count.txt", "edge_count.txt", "placeholder_count.txt",
        "duplicate_nodes.txt", "duplicate_edges.txt"
    )
}

print("connector status:", status["connector"]["state"])
print("connector class:", connector["connector.class"])
print("connector topics:", connector["topics"])
print("raw Neo4j query evidence:")
for name, text in raw.items():
    print(f"[{name}] {text!r}")
print("manifest counts:", neo4j)

assert status["connector"]["state"] == "RUNNING"
assert all(task["state"] == "RUNNING" for task in status["tasks"])
assert connector["topics"] == "cpg.nodes,cpg.edges"
assert "MERGE" in connector["neo4j.cypher.topic.cpg.nodes"]
assert "MERGE" in connector["neo4j.cypher.topic.cpg.edges"]
assert neo4j["duplicate_node_groups"] == neo4j["duplicate_edge_groups"] == 0
assert neo4j["total_nodes"] == neo4j["explicit_nodes"] + neo4j["placeholders"]


connector status: RUNNING
connector class: org.neo4j.connectors.kafka.sink.Neo4jConnector
connector topics: cpg.nodes,cpg.edges
raw Neo4j query evidence:
[node_count.txt] 'node_count\n135198'
[edge_count.txt] 'edge_count\n38141'
[placeholder_count.txt] 'placeholder_count\n1935'
[duplicate_nodes.txt] 'No duplicate nodes found'
[duplicate_edges.txt] 'No duplicate edges found'
manifest counts: {'duplicate_edge_groups': 0, 'duplicate_node_groups': 0, 'edges': 38141, 'explicit_nodes': 133263, 'placeholders': 1935, 'total_nodes': 135198}


## What this chapter proves

| Requirement | Evidence |
|---|---|
| Direct graph route | Connector configuration consumes only node and edge topics. |
| Successful sink | Captured connector and task states are `RUNNING`. |
| Graph materialization | Raw Cypher outputs and manifest counts agree. |
| Idempotence | Duplicate node and edge groups are zero. |

## Final database UI evidence

```{admonition} Database UI evidence
:class: important

Neo4j Browser from the canonical Stage 3 replay final state on `2026-07-23`.
The query filters the replayed `file_id` and `run_id`; it returns 61 target
nodes and 16 target edges after stale cleanup.
```

![Neo4j Browser final state after replay cleanup](../screenshots/replay/neo4j_after_cleanup.png)

## Reflection

```{admonition} Closing reflection
:class: tip

| Point | Result |
|---|---|
| Worked | Kafka Connect reached `RUNNING` and `MERGE` prevented duplicate graph IDs. |
| Failed | The clean Stage 2 run had query files but no dedicated Browser capture. |
| Resolution | Stage 3 added a real hash-validated Neo4j Browser final-state image. |
```
